# TP2 — Segmentation RFM & Clustering Lumina & Co

> **Objectif** : Transformer la base clients nettoyée en segments actionnables. Répondre au CMO : *"Qui sont nos clients ?"* avec des profils comportementaux précis et des recommandations concrètes.

---

## Sommaire
1. [Construction des features RFM enrichies](#etape1)
2. [Scoring RFM & premiers segments](#etape2)
3. [Clustering K-Means](#etape3)
4. [Comparaison RFM vs Clustering](#etape4)
5. [Visualisation & profilage final](#etape5)
6. [Recommandations marketing par segment](#etape6)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 13

# ─── Chargement (base nettoyée du TP1) ────────────────────────────────────────
customers = pd.read_csv('customers.csv')
customers['first_purchase'] = pd.to_datetime(customers['first_purchase'])
customers['last_purchase']  = pd.to_datetime(customers['last_purchase'])

print(f'Base chargée : {len(customers):,} clients | {customers.shape[1]} colonnes')
display(customers.head(3))

Base chargée : 50,000 clients | 9 colonnes


,customer_id,country,first_purchase,last_purchase,n_orders,total_spent,avg_basket,recency_days,tenure_days
0,46995,United Kingdom,2011-08-09 12:20:00,2011-11-16 12:20:00,2.28,57.34,26.44,23.83,99.17
1,22869,United Kingdom,2010-01-11 12:20:00,2011-10-30 12:20:00,46.53,19463.52,356.65,40.92,657.12
2,12593,France,2011-05-05 09:29:00,2011-05-05 09:29:00,1.00,39.50,39.50,218.00,218.00


---
## Étape 1 — Construction des features RFM enrichies <a id='etape1'></a>

### 1.1 Snapshot date & features RFM de base

> **Règle fondamentale** : la date de référence (snapshot) est fixée une fois pour toutes au maximum du dataset — utiliser la date du jour serait une erreur car les données ne vont pas jusqu'à aujourd'hui. Toute la récence serait artificiellement gonflée.

In [2]:
# ─── Snapshot date = max(last_purchase) dans le dataset ──────────────────────
SNAPSHOT_DATE = customers['last_purchase'].max()
print(f'📅 Snapshot date : {SNAPSHOT_DATE.date()}')
print(f'   (Erreur évitée : utiliser pd.Timestamp.today() = {pd.Timestamp.today().date()} '
      f'→ aurait ajouté ~{(pd.Timestamp.today() - SNAPSHOT_DATE).days} jours de récence artificielle)')

# ─── Features RFM de base ─────────────────────────────────────────────────────
df = customers.copy()

# R — Récence : jours depuis le dernier achat (déjà présente dans le CRM)
df['R'] = df['recency_days']

# F — Fréquence : n_orders
df['F'] = df['n_orders']

# M — Montant : total_spent
df['M'] = df['total_spent']

print(f'\nR (récence)   : min={df["R"].min():.0f}j  |  médiane={df["R"].median():.0f}j  |  max={df["R"].max():.0f}j')
print(f'F (fréquence) : min={df["F"].min():.2f}  |  médiane={df["F"].median():.2f}  |  max={df["F"].max():.0f}')
print(f'M (montant)   : min={df["M"].min():.2f}€ |  médiane={df["M"].median():.0f}€  |  max={df["M"].max():,.0f}€')

📅 Snapshot date : 2011-12-09
   (Erreur évitée : utiliser pd.Timestamp.today() = 2026-02-26 → aurait ajouté ~5193 jours de récence artificielle)

R (récence)   : min=0j  |  médiane=168j  |  max=845j
F (fréquence) : min=0.85  |  médiane=2.22  |  max=293
M (montant)   : min=0.16€ |  médiane=100€  |  max=69,631€


### 1.2 Features comportementales enrichies

In [3]:
# ─── Features comportementales supplémentaires ────────────────────────────────

# 1. Panier moyen par commande (avg_basket déjà présent)
df['avg_basket'] = df['avg_basket']

# 2. Ancienneté (tenure) — distincte de la récence
#    Récence = temps depuis DERNIER achat (signal d'engagement récent)
#    Tenure  = durée totale de la relation client (signal de fidélité globale)
df['tenure_days'] = df['tenure_days']

# 3. Cadence d'achat : commandes par 100 jours de relation
#    Normalise la fréquence par la durée de relation — un client avec 10 commandes
#    en 100 jours est plus actif que celui avec 10 commandes en 2 ans
df['cadence'] = df['F'] / (df['tenure_days'].clip(lower=1)) * 100

# 4. Ratio récence/tenure : signal de décrochage
#    Si récence >> tenure : le client a peut-être décroché récemment
#    Si récence << tenure : client actif tout au long de sa relation
df['recency_ratio'] = df['R'] / (df['tenure_days'].clip(lower=1))

# 5. Flag B2B suspect (hérité du TP1)
df['is_b2b'] = (df['M'] > 10_000) | (df['F'] > 100)

# ─── Résumé des features construites ─────────────────────────────────────────
feature_desc = {
    'R'             : 'Récence — jours depuis dernier achat (↓ = mieux)',
    'F'             : 'Fréquence — nombre de commandes',
    'M'             : 'Montant — total dépensé (€)',
    'avg_basket'    : 'Panier moyen par commande (€)',
    'tenure_days'   : 'Ancienneté — durée totale de la relation (jours)',
    'cadence'       : 'Cadence — commandes par 100 jours de relation',
    'recency_ratio' : 'Ratio récence/tenure — signal de décrochage',
}

print('=== FEATURES CONSTRUITES ===')
for feat, desc in feature_desc.items():
    print(f'  {feat:<15} : {desc}')

display(df[list(feature_desc.keys())].describe().round(2))

=== FEATURES CONSTRUITES ===
  R               : Récence — jours depuis dernier achat (↓ = mieux)
  F               : Fréquence — nombre de commandes
  M               : Montant — total dépensé (€)
  avg_basket      : Panier moyen par commande (€)
  tenure_days     : Ancienneté — durée totale de la relation (jours)
  cadence         : Cadence — commandes par 100 jours de relation
  recency_ratio   : Ratio récence/tenure — signal de décrochage


,R,F,M,avg_basket,tenure_days,cadence,recency_ratio
count,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00
mean,238.45,4.75,421.54,65.43,482.48,1.42,0.56
std,218.93,9.55,2014.14,139.87,224.95,4.01,0.42
min,0.00,0.85,0.16,0.16,0.00,0.10,0.00
25%,38.14,1.06,36.99,21.79,343.82,0.32,0.11
50%,168.00,2.22,99.64,38.81,532.98,0.66,0.58
75%,411.51,5.21,280.23,69.10,662.26,1.38,0.96
max,844.59,292.78,69630.66,5876.15,848.62,114.00,1.35
